# Study 890 — Sector Risk-Parity — the teardown

The excess-vs-excess Sharpe race (inverse-vol and ERC vs cap-weight SPY, both minus BIL), the paired block-bootstrap CI on the Sharpe difference, the Newey-West *t* on the mean return difference, the era cut, the costed and levered timers, and the 20-seed synthetic control. All numbers frozen from `docs/results.md`.

In [1]:
R = {'e_start': '2018-10-01', 'e_end': '2026-06-30', 'e_n': 1946, 'e_rebals': 31, 'e_fp': '5e98273e423e', 'e_rows': 2018, 'e_rp_sharpe': 0.572, 'e_rp_ann': 12.73, 'e_rp_vol': 17.74, 'e_rp_dd': -34.7, 'e_spy_sharpe': 0.667, 'e_spy_ann': 15.67, 'e_spy_vol': 19.62, 'e_spy_dd': -33.7, 'e_diff': -0.095, 'e_ci': (-0.302, 0.132), 'e_pneg': 0.81, 'e_nwt': -1.5, 'e_turn': 57, 'e_cost': 1.7, 'e_erc_sharpe': 0.583, 'e_erc_diff': -0.084, 'n_start': '2007-10-01', 'n_end': '2026-06-30', 'n_n': 4716, 'n_rebals': 75, 'n_fp': '2afe10148f92', 'n_rows': 4802, 'n_rp_sharpe': 0.555, 'n_rp_ann': 11.16, 'n_rp_vol': 17.76, 'n_rp_dd': -49.6, 'n_spy_sharpe': 0.553, 'n_spy_ann': 12.29, 'n_spy_vol': 19.88, 'n_spy_dd': -55.2, 'n_diff': 0.002, 'n_ci': (-0.099, 0.11), 'n_pneg': 0.47, 'n_nwt': -1.12, 'n_turn': 48, 'n_cost': 1.4, 'n_erc_sharpe': 0.517, 'n_erc_diff': -0.036, 'era_early': '2007-2015', 'era_early_rp': 0.414, 'era_early_spy': 0.349, 'era_early_diff': 0.064, 'era_early_n': 2079, 'era_late': '2016-2026', 'era_late_rp': 0.695, 'era_late_spy': 0.759, 'era_late_diff': -0.064, 'era_late_n': 2637, 'lev_L': 1.12, 'lev_sharpe': 0.552, 'lev_spy_sharpe': 0.553, 'lev_ann': 10.97, 'lev_spy_ann': 10.99, 'lev_fin': 7.2, 'lev_dd': -54.2, 'cy_2008': 4.6, 'cy_2022': 13.3, 'cy_2023': -14.6, 'cy_2024': -10.6, 'cy_wins': 7, 'cy_years': 20, 'null_mean': 0.0031, 'null_sd': 0.0214, 'null_fire': 0, 'planted': 0.136, 'planted_rp': 1.32, 'planted_spy': 1.18}

## The race — risk-parity vs cap-weight SPY, excess of cash

An unlevered risk-parity book is *expected* to earn less than a tech-heavy cap-weight index, so the fair test is the risk-adjusted Sharpe (and the drawdown), not raw return.

In [2]:
for tag,rp_s,rp_a,rp_v,rp_d,d,ci,p,t in [
  ('11-sector 2018-26', R['e_rp_sharpe'],R['e_rp_ann'],R['e_rp_vol'],R['e_rp_dd'],R['e_diff'],R['e_ci'],R['e_pneg'],R['e_nwt']),
  ('9-sector  2007-26', R['n_rp_sharpe'],R['n_rp_ann'],R['n_rp_vol'],R['n_rp_dd'],R['n_diff'],R['n_ci'],R['n_pneg'],R['n_nwt'])]:
    print(f"{tag}: RP Sharpe {rp_s:.3f} (ann {rp_a:+.1f}% vol {rp_v:.1f}% DD {rp_d:.0f}%)")
    print(f"    Sharpe diff vs SPY {d:+.3f}  95% CI {ci}  P(diff<0)={p:.2f}  NW t(ret diff)={t:+.2f}")
print(f"SPY (11-sec window) Sharpe {R['e_spy_sharpe']:.3f}; SPY (9-sec window) Sharpe {R['n_spy_sharpe']:.3f}")
print(f"ERC variant: 11-sec Sharpe {R['e_erc_sharpe']:.3f} (diff {R['e_erc_diff']:+.3f}), 9-sec {R['n_erc_sharpe']:.3f} (diff {R['n_erc_diff']:+.3f})")

11-sector 2018-26: RP Sharpe 0.572 (ann +12.7% vol 17.7% DD -35%)
    Sharpe diff vs SPY -0.095  95% CI (-0.302, 0.132)  P(diff<0)=0.81  NW t(ret diff)=-1.50
9-sector  2007-26: RP Sharpe 0.555 (ann +11.2% vol 17.8% DD -50%)
    Sharpe diff vs SPY +0.002  95% CI (-0.099, 0.11)  P(diff<0)=0.47  NW t(ret diff)=-1.12
SPY (11-sec window) Sharpe 0.667; SPY (9-sec window) Sharpe 0.553
ERC variant: 11-sec Sharpe 0.583 (diff -0.084), 9-sec 0.517 (diff -0.036)


The bootstrap CI on the Sharpe difference **straddles zero** in every case, and the Newey-West *t* on the mean daily excess-return difference is small and negative — no statistically distinguishable Sharpe advantage on either panel.

## Era cut — the advantage is entirely regime-dependent

In [3]:
print(f"{R['era_early']} (n={R['era_early_n']}): RP {R['era_early_rp']:.3f} vs SPY {R['era_early_spy']:.3f}  diff {R['era_early_diff']:+.3f}  (crisis era: RP wins)")
print(f"{R['era_late']} (n={R['era_late_n']}): RP {R['era_late_rp']:.3f} vs SPY {R['era_late_spy']:.3f}  diff {R['era_late_diff']:+.3f}  (tech bull: RP loses)")
print('A genuine edge holds across sub-eras; this one flips sign — the signature of diversification, not alpha.')

2007-2015 (n=2079): RP 0.414 vs SPY 0.349  diff +0.064  (crisis era: RP wins)
2016-2026 (n=2637): RP 0.695 vs SPY 0.759  diff -0.064  (tech bull: RP loses)
A genuine edge holds across sub-eras; this one flips sign — the signature of diversification, not alpha.


## The timer — can you get paid for it?

Costs first: quarterly rebalancing turns over ~half the book a year, so at 3 bps one-way the drag is ~1–2 bps/yr — trivial. The problem is not costs; it is that the unlevered book earns *less* than SPY. Levering to SPY's vol to chase a return edge just reproduces SPY, because the Sharpe was never higher.

In [4]:
print(f"costs: turnover ~{R['n_turn']}%/yr x 3bps one-way = {R['n_cost']:.1f} bps/yr drag (negligible)")
print(f"levered {R['lev_L']:.2f}x to SPY vol: Sharpe {R['lev_sharpe']:.3f} vs SPY {R['lev_spy_sharpe']:.3f}; "
      f"ann {R['lev_ann']:+.1f}% vs {R['lev_spy_ann']:+.1f}% (financing {R['lev_fin']:.1f} bps/yr, maxDD {R['lev_dd']:.0f}%)")
print('No free lunch: the leverage route lands right back on SPY.')

costs: turnover ~48%/yr x 3bps one-way = 1.4 bps/yr drag (negligible)
levered 1.12x to SPY vol: Sharpe 0.552 vs SPY 0.553; ann +11.0% vs +11.0% (financing 7.2 bps/yr, maxDD -54%)
No free lunch: the leverage route lands right back on SPY.


## Synthetic positive control — the machinery is unbiased

Live: inverse-vol must out-Sharpe the concentrated cap-weight benchmark ONLY when the assets' vols are dispersed, and tie when they are equal.

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from sector_rp import data, strategy as st
nulls = np.array([st.synthetic_detect(data.synthetic_world(vol_spread=0.0, seed=890+s))['sharpe_advantage'] for s in range(8)])
print(f"null (vol_spread=0), 8 seeds: mean advantage {nulls.mean():+.4f} (sd {nulls.std(ddof=1):.4f}), |adv|>0.1 in {(np.abs(nulls)>0.1).sum()}/8")
planted = st.synthetic_detect(data.synthetic_world(vol_spread=0.02, seed=890))
print(f"planted (vol_spread=0.02): advantage {planted['sharpe_advantage']:+.3f} (RP {planted['sr_rp']:.2f} vs cap-weight {planted['sr_bench']:.2f})")

null (vol_spread=0), 8 seeds: mean advantage -0.0028 (sd 0.0185), |adv|>0.1 in 0/8


planted (vol_spread=0.02): advantage +0.136 (RP 1.32 vs cap-weight 1.18)


## Verdict

- **Signal — Mixed.** The claim splits in two. The **drawdown / vol diversification is real and robust**: over 2007–2026 inverse-vol cut volatility to 17.8% (SPY 19.9%) and max drawdown to -50% (SPY -55%), beating SPY by +4.6pp in 2008 and +13.3pp in 2022. But the **excess-of-cash Sharpe advantage does not clear the bar**: it is +0.002 on the long panel (bootstrap 95% CI (-0.099, 0.11) straddles zero, NW *t* = -1.12), -0.095 on the 2018–2026 panel, and it *flips sign* across eras (+0.064 then -0.064). Real risk reduction, no risk-adjusted edge. The 20-seed synthetic control recovers a *planted* advantage cleanly (fires on 0/20 nulls). *Short history on the eleven-sector panel (XLC from 2018-06) is named here.*
- **Tradability — Mirage.** Costs are trivial (1.4 bps/yr), but there is no Sharpe edge to harvest: unlevered you earn *less* than SPY (+11.2% vs +12.3%) for the smoother ride, and levering to SPY's vol reproduces SPY (0.55 vs 0.55 Sharpe). The promised risk-adjusted pickup is a mirage.